# 04 Empirical Shape Model Validation

This notebook evaluates the 15-minute shape layer on observed Dutch quarter-hour DAM data.

Current scope:

- build and compare the mandatory flat-repeat baseline, a simple mean-shape baseline, LEAR, and XGBoost;
- evaluate shape-only performance and reconstructed quarter-hour price performance under the diagnostic/oracle anchor;
- check whether realistic hourly anchors are already available for proper end-to-end Track A validation;
- keep the interpretation explicit: diagnostic/oracle results are useful for shape learning, but they are not yet full realistic forecast results.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Phase 4 Runner

The notebook loads the latest saved Phase 4 artifact by default. Set `RUN_PHASE04 = True` only when you want to rerun the empirical validation from inside the notebook.

In [ ]:
RUN_PHASE04 = False

if RUN_PHASE04:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase04_empirical_validation.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 4 empirical validation failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase04_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 4 artifact exists yet. Run the phase 4 script first.")

checks = pd.read_csv(latest_run / "validation_checks.csv")
realistic_anchor = pd.read_csv(latest_run / "realistic_anchor_availability.csv")
model_config = pd.read_csv(latest_run / "model_configuration_summary.csv")
tuning_results = pd.read_csv(latest_run / "tuning_results.csv")
shape_metrics = pd.read_csv(latest_run / "shape_only_metrics.csv")
price_metrics = pd.read_csv(latest_run / "reconstructed_price_metrics.csv")
performance_by_condition = pd.read_csv(latest_run / "performance_by_condition.csv")
mae_by_hour = pd.read_csv(latest_run / "mae_by_hour.csv")
recommended_model = pd.read_csv(latest_run / "recommended_model_summary.csv")
predictions = pd.read_csv(latest_run / "predictions_long.csv")
example_days = pd.read_csv(latest_run / "example_days.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase04_run": str(latest_run)}]))

## Validation Checks

In [ ]:
display(checks)

## Realistic Hourly Anchor Availability

In [ ]:
display(realistic_anchor)

## Model Configuration Summary

In [ ]:
display(model_config)

## Tuning Summary

In [ ]:
display(tuning_results)

## Shape-Only Metrics

In [ ]:
display(shape_metrics)

## Reconstructed Quarter-Hour Price Metrics

In [ ]:
display(price_metrics)

## Main Comparison Plot

Lower MAE is better. The diagnostic/oracle anchor means the hourly level is fixed to the observed hourly mean, so this comparison isolates the quarter-hour shape layer.

In [ ]:
plot_frame = price_metrics[price_metrics["dataset_split"].astype(str) == "test"].copy()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(plot_frame["model"], plot_frame["mae"])
ax.set_title("Test MAE Under Diagnostic/Oracle Anchor")
ax.set_ylabel("MAE (EUR/MWh)")
ax.set_xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Error By Hour Of Day

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for model_name, group in mae_by_hour.groupby("model"):
    ax.plot(group["local_hour_of_day"], group["mae"], marker="o", label=model_name)
ax.set_title("Test MAE By Hour Of Day")
ax.set_xlabel("Local hour of day")
ax.set_ylabel("MAE (EUR/MWh)")
ax.legend()
plt.tight_layout()
plt.show()

## Performance By Condition

In [ ]:
display(performance_by_condition.head(60))

## Recommended Model Summary

In [ ]:
display(recommended_model)

## Example Days

In [ ]:
display(example_days)

## Selected Day Forecast Comparison

The plot below uses the first available example day from the saved artifact bundle.

In [ ]:
if not example_days.empty:
    selected_day = example_days.iloc[0]["hour_local_date"]
    day_slice = predictions[predictions["hour_local_date"].astype(str) == str(selected_day)].copy()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    actual = day_slice.drop_duplicates(subset=["timestamp_utc"])[["timestamp_local", "price_eur_per_mwh"]].sort_values("timestamp_local")
    ax.plot(pd.to_datetime(actual["timestamp_local"]), actual["price_eur_per_mwh"], label="actual", linewidth=2.0)
    for model_name in ["flat_repeat", "lear_shape", "xgboost_shape"]:
        model_slice = (
            day_slice[day_slice["model"].astype(str) == model_name][["timestamp_local", "predicted_price_eur_per_mwh"]]
            .sort_values("timestamp_local")
        )
        if not model_slice.empty:
            ax.plot(pd.to_datetime(model_slice["timestamp_local"]), model_slice["predicted_price_eur_per_mwh"], label=model_name)
    ax.set_title(f"Quarter-Hour Prices On {selected_day}")
    ax.set_xlabel("Local timestamp")
    ax.set_ylabel("Price (EUR/MWh)")
    ax.legend()
    plt.tight_layout()
    plt.show()